# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 14
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 14


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
# Define base path and filenames
base_path = f'./{seed}'
filenames = [f'{base_path}/mHemi_Brain_Splane{i}.h5ad' for i in range(3)]

# Read each file into a list of AnnData objects
ad_list = [sc.read(f) for f in filenames]

# Combine them into one AnnData
adata = ad.concat(ad_list, join='outer', label='batch', keys=[f'Splane{i}' for i in range(3)])

adata[0]

View of AnnData object with n_obs × n_vars = 1 × 1122
    obs: 'brain_section_label', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'cluster_alias', 'z', 'subclass_confidence_score', 'cluster_confidence_score', 'high_quality_transfer', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'original_clusters', '01 IT-ET Glut', '02 NP-CT-L6b Glut', '03 OB-CR Glut', '04 DG-IMN Glut', '05 OB-IMN GABA', '06 CTX-CGE GABA', '07 CTX-MGE GABA', '08 CNU-MGE GABA', '09 CNU-LGE GABA', '10 LSX GABA', '11 CNU-HYa GABA', '12 HY GABA', '13 CNU-HYa Glut', '14 HY Glut', '15 HY Gnrh1 Glut', '16 HY MM Glut', '17 MH-LH Glut', '18 TH Glut', '19 MB Glut', '20 MB GABA', '21 MB Dopa', '22 MB-HB Sero', '23 P Glut', '24 MY Glut', '25 Pineal Glut', '26 P GABA', '27 MY GABA', '28 CB GABA', '29 CB Glut', '30 Astro-Epen', '31 OPC-Oligo', '32 OEC', '33 Vascular', '34 Immune', 'spatial_doma

In [5]:
adata.obs['brain_section_label'].unique()

['Zhuang-ABCA-1.010', 'Zhuang-ABCA-1.011', 'Zhuang-ABCA-1.012', 'Zhuang-ABCA-1.013', 'Zhuang-ABCA-1.014', ..., 'Zhuang-ABCA-1.137', 'Zhuang-ABCA-1.138', 'Zhuang-ABCA-1.139', 'Zhuang-ABCA-1.140', 'Zhuang-ABCA-1.141']
Length: 129
Categories (129, object): ['Zhuang-ABCA-1.010', 'Zhuang-ABCA-1.011', 'Zhuang-ABCA-1.012', 'Zhuang-ABCA-1.013', ..., 'Zhuang-ABCA-1.138', 'Zhuang-ABCA-1.139', 'Zhuang-ABCA-1.140', 'Zhuang-ABCA-1.141']

In [6]:
adata_list=[]
slices = adata.obs['brain_section_label'].unique().tolist()

for i in range(129):
    adata_list.append(adata[adata.obs['brain_section_label'] == slices[i]])
#     adata_list[i].uns['spatial_domain_colors'] = pd.Series(np.unique(adata_list[i].obs.spatial_domain)).replace(dict(zip([4,1,6,7,2,3,5,0],['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f']))).values.flatten()
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Alignment slice 2 to 1
Alignment slice 3 to 2
Alignment slice 4 to 3
Alignment slice 5 to 4
Alignment slice 6 to 5
Alignment slice 7 to 6
Alignment slice 8 to 7
Alignment slice 9 to 8
Alignment slice 10 to 9
Alignment slice 11 to 10
Alignment slice 12 to 11
Alignment slice 13 to 12
Alignment slice 14 to 13
Alignment slice 15 to 14
Alignment slice 16 to 15
Alignment slice 17 to 16
Alignment slice 18 to 17
Alignment slice 19 to 18
Alignment slice 20 to 19
Alignment slice 21 to 20
Alignment slice 22 to 21
Alignment slice 23 to 22
Alignment slice 24 to 23
Alignment slice 25 to 24
Alignment slice 26 to 25
Alignment slice 27 to 26
Alignment slice 28 to 27
Alignment slice 29 to 28
Alignment slice 30 to 29
Alignment slice 31 to 30
Alignment slice 32 to 31
Alignment slice 33 to 32
Alignment slice 34 to 33
Alignment slice 35 to 34
Alignment slice 36 to 35
Alignment slice 37 to 36
Alignment slice 38 to 37
Alignment slice 39 to 38
Alignment slice 40 to 39


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

In [9]:
adata_list[0].obs['class'].unique()

['05 OB-IMN GABA', '13 CNU-HYa Glut', '34 Immune', '32 OEC', '33 Vascular', ..., '06 CTX-CGE GABA', '10 LSX GABA', '12 HY GABA', '02 NP-CT-L6b Glut', '29 CB Glut']
Length: 14
Categories (14, object): ['02 NP-CT-L6b Glut', '03 OB-CR Glut', '05 OB-IMN GABA', '06 CTX-CGE GABA', ..., '31 OPC-Oligo', '32 OEC', '33 Vascular', '34 Immune']

In [10]:
adata_list[0].obs['original_clusters'].unique()

['GABA', 'Glut', 'NaN', 'Dopa', 'Glut-GABA']
Categories (5, object): ['Dopa', 'GABA', 'Glut', 'Glut-GABA', 'NaN']

## LAA & ARI

In [11]:
from sklearn.metrics import pairwise
from sklearn.metrics import adjusted_rand_score

def cal_layer_based_alignment_result_mHypothalamus(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['original_clusters'])
    labels.extend(s2.obs['original_clusters'])

    res = []
    #l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    ir = 0
    max_smallerThan0 = []
    sum_smallerThan0 = []
    min_smallerThan0 = []

    for i, elem in enumerate(alignment):
        #if np.sum(elem)>0:
        if np.max(elem) - np.min(elem) != 0:
            if labels[i] == labels[elem.argmax() + alignment.shape[0]]:
                cnt0 += 1
        else:
            ir += 1
            max_smallerThan0.append(np.max(elem))
            sum_smallerThan0.append(np.sum(elem))
            min_smallerThan0.append(np.min(elem))
    print("the number of sum alignment equals 0:", str(ir))
    if len(sum_smallerThan0)>0:
        print("sum in sum alignment equals 0 is between", np.min(sum_smallerThan0), np.max(sum_smallerThan0))
        print("max in sum alignment equals 0 is between", np.min(max_smallerThan0), np.max(max_smallerThan0))
        print("min in sum alignment equals 0 is between", np.min(min_smallerThan0), np.max(min_smallerThan0))

    res.extend([cnt0])
    return res


def compute_alignment_ari(pi, sliceA, sliceB):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['original_clusters'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['original_clusters']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['original_clusters'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['original_clusters'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

In [ ]:
ari_values = []
laa_values = []

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    alignment = pairwise.euclidean_distances(f1, f2)
    alignment = np.exp(-alignment)
    
    name1 = slices[target_ind].split('.')[-1]
    name2 = slices[source_ind].split('.')[-1]
    # Save as .pkl
#     alignment_df = pd.DataFrame(alignment)
#     alignment_df.to_pickle(f'./{seed}/{name1}_{name2}_AlignmentPi.pkl')
    
    res = cal_layer_based_alignment_result_mHypothalamus(alignment, adata_list[target_ind], adata_list[source_ind])
    laa = res[0]/len(adata_list[source_ind].obs['original_clusters'])
    ari = compute_alignment_ari(alignment, adata_list[target_ind], adata_list[source_ind])
    ari_values.append(ari)
    laa_values.append(laa)
    print(laa, ari)

# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame([laa_values], columns=header_row)

the number of sum alignment equals 0: 0
0.5164560294892049 0.158529313335653
the number of sum alignment equals 0: 0
0.40323383084577114 0.15943166583394192
the number of sum alignment equals 0: 0
0.322423533995714 0.10342567298192124
the number of sum alignment equals 0: 0
0.49467893217893216 0.17659568693694772
the number of sum alignment equals 0: 0
0.506539190042392 0.19049147404957775
the number of sum alignment equals 0: 0
0.5728342245989305 0.15616800321926028
the number of sum alignment equals 0: 0
0.509829619921363 0.16063292098626228
the number of sum alignment equals 0: 0
0.3064575307283067 0.14931645301622376
the number of sum alignment equals 0: 0
0.6542398584607608 0.11420404030876426
the number of sum alignment equals 0: 0
0.26993865030674846 0.11548419919403632
the number of sum alignment equals 0: 0
0.5188568327756172 0.13471068882585366
the number of sum alignment equals 0: 0
0.4556202087374157 0.17413055512734668
the number of sum alignment equals 0: 0
0.564575645756

In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/mHemi_brain_ARI.csv')
laa_df.to_csv(f'./{seed}/mHemi_brain_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [14]:
c_dict = {
    'Chol': '#9e0142',
    'Dopa': '#d8434e',
    'GABA': '#f67a49',
    'GABA-Glyc': '#fdbf6f',
    'Glut': '#feeda1',
    'Glut-GABA': '#f1f9a9',
    'Hist': '#bfe5a0',
    'NaN': '#74c7a5',
    'Nora': '#378ebb',
    'Sero': '#5e4fa2'
}

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = slices[i].split('.')[-1]
    loc['original_clusters'] = adata_list[i].obs['original_clusters']
    loc['celltype_colors'] = adata_list[i].obs['original_clusters'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_mHemi_aligned_coordinates_colors.csv')

In [16]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
cell_label,,,,,,
196289950361762737219003647068046468794,0.901972,-0.559732,0,010,GABA,#f67a49
20790876773804717808797081150482960179,0.906682,-0.669188,0,010,Glut,#feeda1
88789079621616508498928319995803668262,0.905021,-0.648624,0,010,NaN,#74c7a5
100937236530311940806885443461628031796,0.890939,-0.433031,0,010,NaN,#74c7a5
103229474276965460653698200970306892913,0.964986,-0.565069,0,010,GABA,#f67a49
...,...,...,...,...,...,...
56875541036959821743348544221361587986,1.738188,-3.500206,128,141,GABA,#f67a49
60379927107929501802082916259496260324,1.612279,-3.608258,128,141,GABA,#f67a49
63409849943414630555243725901159951596,1.801894,-3.541199,128,141,GABA,#f67a49


Plot all aligned slices stacked in 3D constructed by Scube

In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
seed = 4
coo = pd.read_csv(f'./{seed}/SPACEL_mHemi_aligned_coordinates_colors.csv')


In [ ]:
crop = coo[['X', 'Y', 'Z']]
fig = plot_3d(crop.values, val=None, color=coo['celltype_colors'], elev=180, azim=0, roll=90, frameon=False, show=True, s=1, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_mHemi_3dviz_angle180_0_90_s=1.png', bbox_inches="tight")